# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR² dataset on second primary colorectal cancer, using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json), which provides FAIR, machine-readable access to tabular data and metadata.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access basic metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Identifier: {meta.identifier}\nVersion: {meta.version}")
print(f"Fields containing personal/sensitive information: {meta.personalSensitiveInformation}")
print(f"Date published: {meta.datePublished}")


## 2. Data Overview

Let's examine available record sets in the dataset. Each entity (record set, field, column) is referenced by its unique `@id`.

In [ ]:
# List all available record sets and their fields using `@id`
record_sets = dataset.metadata.record_sets  # list of mlcroissant.RecordSet

print("Available record sets:")
for rs in record_sets:
    print(f"  Record Set '@id': {rs.id} (name: {rs.name})")
    if hasattr(rs, 'fields'):
        print("    Fields (by @id):")
        for field in rs.fields:
            print(f"     - {field.id}: {field.name} (type: {field.data_type})")
    else:
        print("    No fields found.")

## 3. Data Extraction

We load tables from each record set into separate Pandas DataFrames. All lookups use the entity `@id`. For this dataset, there's typically a main record set containing the tabular data (e.g. clinical variables).

In [ ]:
# Extract all record sets into DataFrames keyed by record set @id
dataframes = {}

# If you know a specific record set @id, you can use it here. Otherwise, extract all:
for rs in record_sets:
    # The record set's @id is used for querying
    rs_id = rs.id
    print(f"Loading record set {rs_id} ...")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} records, columns: {list(df.columns)}")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

# For this dataset, print the record set keys and show the first five rows of the main one
main_rs_id = next(iter(dataframes.keys()))  # Use the first loaded as main (usually only one in clinical datasets)
print("\nMain record set (first record set):", main_rs_id)
print(f"Columns: {dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll perform some analysis and transformations. All columns are referenced by their Croissant `@id`.

Typical numeric columns may include age or diagnosis intervals. We check for a numeric field, filter based on a threshold, normalize, and group by a categorical attribute (e.g., sex or cancer subtype).

In [ ]:
df = dataframes[main_rs_id]

# Try to select a numeric field (e.g., look for "age" or "interval" in columns)
numeric_field = None
for col in df.columns:
    col_lower = col.lower()
    if any(word in col_lower for word in ["age", "interval", "duration", "years", "months"]):
        numeric_field = col
        break
if numeric_field is None:
    # fallback: pick the first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
assert numeric_field is not None, "Could not find a numeric field for demonstration."

print(f"Analyzing numeric field (by @id): {numeric_field}")

# Filter records with values above a threshold
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.1f} (mean): {len(filtered_df)} rows")

# Normalize
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Try grouping by a common categorical field, such as sex or anatomical location
group_field = None
for c in df.columns:
    if any(word in c.lower() for word in ["sex", "gender", "site", "location", "type", "anatomical"]):
        group_field = c
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and, if possible, compare across groups (e.g., by sex or anatomical location).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

if group_field:
    plt.figure(figsize=(9,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} grouped by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

We've explored the main record set of the FAIR² dataset describing clinicopathological and molecular features of second primary colorectal cancer. Using `mlcroissant`, we loaded tabular data by record set and field `@id`, inspected the schema, and performed basic EDA including filtering and grouping. You can now proceed with further biomedical or ML analyses, informed by explicit schema information.